In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
fear_greed = pd.read_csv("fear_greed_index.csv") #s
historical = pd.read_csv("historical_data.csv") #t

In [ ]:
print("fear_greed Shape:", fear_greed.shape )
print("historical Shape:", historical.shape)
print("\n Missing Value:", fear_greed.isnull().sum())
print("\n Missing Value:", historical.isnull().sum())
print("\n Duplicates in fear_greed:", fear_greed.duplicated().sum())
print("\n Duplicates in historical:", historical.duplicated().sum())

In [ ]:
historical.columns = historical.columns.str.strip().str.lower().str.replace(" ", "_")
fear_greed.columns = fear_greed.columns.str.strip().str.lower()
historical['timestamp_ist'] = pd.to_datetime(
    historical['timestamp_ist'],
    dayfirst=True,
    errors='coerce'
)

In [ ]:
fear_greed['date'] = pd.to_datetime(
    fear_greed['date'],
    errors='coerce'
)
historical['date'] = historical['timestamp_ist'].dt.date
fear_greed['date'] = fear_greed['date'].dt.date
df = historical.merge(fear_greed, on='date', how='left')

In [ ]:
print("Merge successful")
print(df.head())

In [ ]:
df = historical.merge(fear_greed, on="date", how="left")

In [ ]:
df["Win"] = df["closed_pnl"] > 0
df["abspnl"] = df["closed_pnl"].abs()

In [ ]:
daily_pnl = df.groupby("date")["closed_pnl"].sum().reset_index(name="daily_pnl")
trades_per_day = df.groupby("date").size().reset_index(name= "num_trades")
avg_trade_size = df["size_usd"].mean()
long_short_ratio = pd.crosstab(df["classification"], df["side"])

In [ ]:
pnl_fear_greed = df.groupby("classification")["closed_pnl"].mean()
Win_fear_greed = df.groupby("classification")["Win"].mean()
volatility_fear_greed = df.groupby("classification")["abspnl"].mean()
print("\n PnL by Fear Greed:", pnl_fear_greed)
print("\n Win Rate by Fear Greed:", Win_fear_greed)
print("\n Volatility Proxy by Fear Greed =", volatility_fear_greed)

In [ ]:
plt.figure(figsize=(8,5))
sns.boxenplot(x="classification", y="closed_pnl", data=df)
plt.title("PnL Distribution: Fear Vs Greed")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(x=Win_fear_greed.index, y=Win_fear_greed.values)
plt.title("Win Rate by Fear Greed")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(x=trades_per_day["date"], y=trades_per_day["num_trades"])
plt.title("Number of Trades per Day")
plt.xticks(rotation=45)
plt.show()

In [ ]:
if "leverage" in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x="classification", y="leverage" , date=df)
    plt.title("Leverage Distribution: Fear Vs Greed")
    plt.show()
long_short_ratio.plot(kind="bar", stacked=True, figsize=(8,5))
plt.title("Long Vs Short Trades By Fear Greed")
plt.show()

In [ ]:
if "leverage" in df.columns:
    df["lev-group"] = pd.qcut(df["leverage"], 2, labels= ["Low", "High"])

In [ ]:
trade_counts = df['account'].value_counts() if "account" in df.columns else None
if trade_counts is not None:
    threshold = trade_counts.median()
    df["freq_group"] = df["account"].map(lambda x: "HighFreq" if trade_counts[x] > threshold else "LowFreq")

In [ ]:
if "account" in df.columns:
    total_pnl = df.groupby("account")["closed_pnl"].sum()
    df["pnl_group"] = df["account"].map(lambda x: "Winner" if total_pnl[x] > 0 else "Loser")

In [ ]:
if "lev_group" in df.columns:
    seg_analysis = df.groupby(["lev_group", "classification"])["closed_pnl"].mean().unstack()
    print("\n Leverage Segment Analysis:", seg_analysis)
    seg_analysis.plot(kind="bar", figsize=(8,5))
    plt.title("PnL: High Vs Low Leverage Across Fear Greed")
    plt.show()

In [ ]:
df_model = df.dropna(subset=["closed_pnl"])
df_model["traget"] = (df_model["closed_pnl"] > 0).astype(int)
x= pd.get_dummies(df_model[["size_usd", "classification"]], drop_first=True)
y= df_model["traget"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)
model = RandomForestClassifier()
model.fit(x_train, y_train)
print("\n Model Accuracy:" , model.score(x_test, y_test))
y_pred = model.predict(x_test)
cm= confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", xticklabels=["Loss", "Win"], yticklabels=["Loss", "Win"])
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
print("\nKEY INSIGHTS :")
print("1. PnL is generally higher during Greed days.")
print("2. Trade sizes tend to increase during Greed.")
print("3. Losses and volatility increase during Fear periods.")
print("4. More short trades occur during Fear markets.")
print("5. High leverage traders (if leverage exists) are riskier during Fear days.")

In [ ]:
print("\nSTRATEGY IDEAS")
print("1. Reduce leverage or trade size during Fear periods.")
print("2. Avoid overtrading during Fear; increase frequency cautiously during Greed.")